# LSTAR e ESTAR: Transicao Suave - SOLUTION

Neste notebook exploramos modelos **Smooth Transition Autoregressive (STAR)**,
que generalizam os modelos TAR permitindo uma transicao **gradual** entre regimes.

## Conteudo
1. Funcao de transicao logistica
2. LSTAR - Logistic STAR
3. Funcao de transicao exponencial
4. ESTAR - Exponential STAR
5. LSTAR vs ESTAR - quando usar cada um
6. Parametro de suavidade $\gamma$

## Referencias
- Terasvirta, T. (1994). Specification, estimation, and evaluation of smooth transition autoregressive models. *Journal of the American Statistical Association*, 89, 208-218.
- Luukkonen, R., Saikkonen, P., & Terasvirta, T. (1988). Testing linearity against smooth transition autoregressive models. *Biometrika*, 75, 491-499.
- Granger, C.W.J. & Terasvirta, T. (1993). *Modelling Nonlinear Economic Relationships*. Oxford University Press.
- van Dijk, D., Terasvirta, T. & Franses, P.H. (2002). Smooth transition autoregressive models - a survey of recent developments. *Econometric Reviews*, 21, 1-47.

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from archbox.threshold import (
    ESTAR,
    LSTAR,
    exponential_transition,
    logistic_transition,
    plot_transition,
)

# Configuracao de graficos
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
plt.style.use('seaborn-v0_8-whitegrid')

# Carregar dados
data = pd.read_csv('../data/sp500_returns.csv', parse_dates=['date'], index_col='date')
y = data['y'].values
print(f'S&P500 Returns: {len(y)} observacoes')
print(f'Media: {y.mean():.4f}, Desvio-padrao: {y.std():.4f}')
data.head()

## 1. Funcao de transicao logistica

A funcao de transicao **logistica** e dada por:

$$
G(s_t; \gamma, c) = \frac{1}{1 + \exp(-\gamma(s_t - c))}
$$

onde:
- $s_t$ e a variavel de transicao (tipicamente $y_{t-d}$)
- $\gamma > 0$ e o **parametro de suavidade**
- $c$ e o **parametro de localizacao** (threshold)

Propriedades:
- $G \in [0, 1]$
- $G = 0.5$ quando $s_t = c$
- $\gamma \to \infty$: transicao abrupta (equivale a TAR)
- $\gamma \to 0$: modelo linear (sem transicao)

A funcao logistica e **assimetrica** em relacao a $c$: o comportamento difere
dependendo de $s_t < c$ vs $s_t > c$.

In [ ]:
# Funcao logistica para diferentes valores de gamma

s = np.linspace(-3, 3, 500)
c = 0.0  # threshold centralizado

gamma_values = [0.5, 1, 2, 5, 10, 50]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Funcao de transicao logistica
for gamma in gamma_values:
    G = logistic_transition(s, gamma, c)
    axes[0].plot(s, G, linewidth=2, label=f'$\\gamma = {gamma}$')

axes[0].axhline(0.5, color='gray', linestyle=':', alpha=0.5)
axes[0].axvline(c, color='gray', linestyle=':', alpha=0.5)
axes[0].set_xlabel('$s_t$', fontsize=12)
axes[0].set_ylabel('$G(s_t; \\gamma, c)$', fontsize=12)
axes[0].set_title('Funcao de Transicao Logistica', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].set_ylim(-0.05, 1.05)

# Usando a funcao plot_transition do archbox
fig_archbox = plot_transition(s, gamma_values=[1, 5, 50], c=0.0, transition_type='logistic')

plt.tight_layout()
plt.savefig('../outputs/logistic_transition.png', bbox_inches='tight')
plt.show()

## 2. LSTAR - Logistic Smooth Transition AR

O modelo **LSTAR** e definido como:

$$
y_t = (\phi_0^{(1)} + \phi_1^{(1)} y_{t-1})(1 - G(s_t; \gamma, c)) + (\phi_0^{(2)} + \phi_1^{(2)} y_{t-1})G(s_t; \gamma, c) + \varepsilon_t
$$

Equivalentemente:
$$
y_t = \boldsymbol{x}_t'\boldsymbol{\phi}^{(1)} + \boldsymbol{x}_t'\boldsymbol{\phi}^{(2)} G(s_t; \gamma, c) + \varepsilon_t
$$

O LSTAR e adequado para capturar **assimetrias**: por exemplo, quando a dinamica
em recessao difere da dinamica em expansao.

In [ ]:
# Estimacao LSTAR com archbox

model_lstar = LSTAR(y, order=1, delay=1, gamma_grid=50, c_grid=50)
results_lstar = model_lstar.fit()

print(results_lstar.summary())
print(f'\nThreshold (c): {results_lstar.threshold:.4f}')
print(f'Gamma: {results_lstar.transition_params.get("gamma", "N/A")}')
print(f'\nParametros Regime 1: {results_lstar.params_regime1}')
print(f'Parametros Regime 2: {results_lstar.params_regime2}')
print(f'AIC: {results_lstar.aic:.4f}')
print(f'BIC: {results_lstar.bic:.4f}')

## 3. Funcao de transicao exponencial

A funcao de transicao **exponencial** e dada por:

$$
G(s_t; \gamma, c) = 1 - \exp(-\gamma(s_t - c)^2)
$$

Propriedades:
- $G \in [0, 1]$
- $G = 0$ quando $s_t = c$ (minimo no centro)
- $G \to 1$ quando $|s_t - c| \to \infty$
- **Simetrica** em torno de $c$

A funcao exponencial e util quando o comportamento e similar para valores
extremos (tanto altos quanto baixos) de $s_t$, mas difere no centro.

In [ ]:
# Funcao exponencial para diferentes valores de gamma

s = np.linspace(-3, 3, 500)
c = 0.0

gamma_values_exp = [0.5, 1, 2, 5, 10, 50]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Funcao de transicao exponencial
for gamma in gamma_values_exp:
    G = exponential_transition(s, gamma, c)
    axes[0].plot(s, G, linewidth=2, label=f'$\\gamma = {gamma}$')

axes[0].axhline(0, color='gray', linestyle=':', alpha=0.5)
axes[0].axvline(c, color='gray', linestyle=':', alpha=0.5)
axes[0].set_xlabel('$s_t$', fontsize=12)
axes[0].set_ylabel('$G(s_t; \\gamma, c)$', fontsize=12)
axes[0].set_title('Funcao de Transicao Exponencial', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].set_ylim(-0.05, 1.05)

# Comparacao logistica vs exponencial
gamma_comp = 5
G_log = logistic_transition(s, gamma_comp, c)
G_exp = exponential_transition(s, gamma_comp, c)
axes[1].plot(s, G_log, 'b-', linewidth=2, label=f'Logistica ($\\gamma={gamma_comp}$)')
axes[1].plot(s, G_exp, 'r--', linewidth=2, label=f'Exponencial ($\\gamma={gamma_comp}$)')
axes[1].axvline(c, color='gray', linestyle=':', alpha=0.5)
axes[1].set_xlabel('$s_t$', fontsize=12)
axes[1].set_ylabel('$G(s_t; \\gamma, c)$', fontsize=12)
axes[1].set_title('Logistica vs Exponencial', fontsize=13)
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.savefig('../outputs/exponential_transition.png', bbox_inches='tight')
plt.show()

## 4. ESTAR - Exponential Smooth Transition AR

O modelo **ESTAR** usa a funcao de transicao exponencial:

$$
y_t = \boldsymbol{x}_t'\boldsymbol{\phi}^{(1)} + \boldsymbol{x}_t'\boldsymbol{\phi}^{(2)} \left[1 - \exp(-\gamma(s_t - c)^2)\right] + \varepsilon_t
$$

O ESTAR e adequado para modelar:
- **Reversao a media**: desvios grandes (em qualquer direcao) de $c$ levam
  a dinamica diferente do que desvios pequenos
- Taxas de cambio (teoria de paridade do poder de compra)
- Taxas de juros (reversao a media nao-linear)

In [ ]:
# Estimacao ESTAR com archbox

model_estar = ESTAR(y, order=1, delay=1, gamma_grid=50, c_grid=50)
results_estar = model_estar.fit()

print(results_estar.summary())
print(f'\nThreshold (c): {results_estar.threshold:.4f}')
print(f'Gamma: {results_estar.transition_params.get("gamma", "N/A")}')
print(f'\nParametros Regime 1: {results_estar.params_regime1}')
print(f'Parametros Regime 2: {results_estar.params_regime2}')
print(f'AIC: {results_estar.aic:.4f}')
print(f'BIC: {results_estar.bic:.4f}')

## 5. LSTAR vs ESTAR - quando usar cada um

| Caracteristica | LSTAR | ESTAR |
|---------------|-------|-------|
| Funcao $G$ | Logistica | Exponencial |
| Simetria | **Assimetrica** | **Simetrica** |
| $G$ no centro ($s_t = c$) | 0.5 | 0 |
| Regimes extremos | Diferentes | Similares |
| Aplicacao tipica | Ciclos economicos | Reversao a media |

**Regra pratica (Terasvirta, 1994):**
- Use **LSTAR** quando a dinamica difere entre $s_t < c$ e $s_t > c$ (assimetria)
- Use **ESTAR** quando a dinamica e similar nos extremos mas diferente no centro (simetria)

A **sequencia de testes de Terasvirta** (ver Notebook 03) ajuda a escolher formalmente.

In [ ]:
# Comparacao LSTAR vs ESTAR nos mesmos dados

print('=== Comparacao LSTAR vs ESTAR ===')
print(f'{"Metrica":<25} {"LSTAR":>12} {"ESTAR":>12}')
print('-' * 50)
print(f'{"Log-likelihood":<25} {results_lstar.loglike:>12.4f} {results_estar.loglike:>12.4f}')
print(f'{"AIC":<25} {results_lstar.aic:>12.4f} {results_estar.aic:>12.4f}')
print(f'{"BIC":<25} {results_lstar.bic:>12.4f} {results_estar.bic:>12.4f}')
print(f'{"Threshold (c)":<25} {results_lstar.threshold:>12.4f} {results_estar.threshold:>12.4f}')
gamma_l = results_lstar.transition_params.get('gamma', float('nan'))
gamma_e = results_estar.transition_params.get('gamma', float('nan'))
print(f'{"Gamma":<25} {gamma_l:>12.4f} {gamma_e:>12.4f}')
print()

# Plotar funcoes de transicao estimadas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Transicao LSTAR
idx_sort = np.argsort(results_lstar.transition_values) if len(results_lstar.transition_values) > 0 else []
axes[0].plot(results_lstar.transition_values, 'b-', alpha=0.7)
axes[0].set_title('LSTAR: G(s_t) ao longo do tempo', fontsize=13)
axes[0].set_xlabel('t', fontsize=12)
axes[0].set_ylabel('$G(s_t)$', fontsize=12)
axes[0].set_ylim(-0.05, 1.05)

# Transicao ESTAR
axes[1].plot(results_estar.transition_values, 'r-', alpha=0.7)
axes[1].set_title('ESTAR: G(s_t) ao longo do tempo', fontsize=13)
axes[1].set_xlabel('t', fontsize=12)
axes[1].set_ylabel('$G(s_t)$', fontsize=12)
axes[1].set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.savefig('../outputs/lstar_vs_estar_transition.png', bbox_inches='tight')
plt.show()

melhor = 'LSTAR' if results_lstar.aic < results_estar.aic else 'ESTAR'
print(f'=> Modelo preferido por AIC: {melhor}')

## 6. Parametro de suavidade $\gamma$

O parametro $\gamma$ controla a velocidade da transicao entre regimes:

- **$\gamma$ pequeno** ($\approx 0$): transicao muito suave $\to$ modelo quase linear
- **$\gamma$ moderado**: transicao gradual entre regimes
- **$\gamma$ grande** ($\to \infty$): transicao abrupta $\to$ equivale a TAR/SETAR

Na pratica, $\gamma$ e frequentemente normalizado pelo desvio-padrao de $s_t$
para facilitar a comparacao entre series diferentes:

$$
\tilde{\gamma} = \frac{\gamma}{\hat{\sigma}_s}
$$

In [ ]:
# Efeito de gamma na funcao de transicao logistica

# Demonstrar o efeito de gamma na funcao de transicao
s_range = np.linspace(y.min(), y.max(), 300)
c_est = results_lstar.threshold

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
gamma_demo = [0.5, 1, 2, 5, 10, 100]

for i, gamma in enumerate(gamma_demo):
    ax = axes[i // 3, i % 3]
    G = logistic_transition(s_range, gamma, c_est)
    ax.plot(s_range, G, 'b-', linewidth=2)
    ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
    ax.axvline(c_est, color='red', linestyle='--', alpha=0.7)
    ax.set_title(f'$\\gamma = {gamma}$', fontsize=12)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel('$s_t$')
    ax.set_ylabel('$G(s_t)$')

plt.suptitle(f'Efeito de $\\gamma$ na funcao de transicao logistica (c = {c_est:.3f})',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../outputs/gamma_effect.png', bbox_inches='tight')
plt.show()

print(f'Gamma estimado pelo LSTAR: {gamma_l:.4f}')
print(f'Threshold estimado: {c_est:.4f}')
print(f'Desvio-padrao de s_t: {y[:-1].std():.4f}')
print(f'Gamma normalizado: {gamma_l / y[:-1].std():.4f}')

## Resumo

| Modelo | Funcao $G$ | Simetria | $\gamma \to \infty$ |
|--------|-----------|----------|--------------------|
| LSTAR  | $\frac{1}{1+e^{-\gamma(s-c)}}$ | Assimetrica | TAR |
| ESTAR  | $1 - e^{-\gamma(s-c)^2}$ | Simetrica | Threshold duplo |

**Pontos-chave:**
- STAR generaliza TAR com transicao suave controlada por $\gamma$
- LSTAR: assimetrica (ciclos economicos, crescimento vs recessao)
- ESTAR: simetrica (reversao a media, taxas de cambio)
- $\gamma$ grande $\to$ TAR; $\gamma$ pequeno $\to$ linear
- Terasvirta (1994) propoe uma sequencia de testes para escolher entre LSTAR e ESTAR

No proximo notebook, veremos os **testes formais de linearidade** que ajudam
a decidir se um modelo nao-linear e necessario e qual tipo usar.